# Within-Day Double Differences
## An internal comparison within the same day

The design compares the school-arrival period, 07:00–09:00, with mid-morning, 10:00–12:00. Both windows share the same date and monitoring unit, which helps remove common seasonal, meteorological and city-wide factors:

    gap(date, monitoring unit) = log(arrival concentration) - log(comparison concentration)
    gap ~ school term + controls

The coefficient measures how much the morning gap WIDENS during school terms. Multiplicative factors common to both windows cancel; factors that evolve differently during the morning do not necessarily cancel. This is the collapsed version of a stacked two-window panel with date-by-unit fixed effects: the first difference is within a date–unit cell, and beta compares those gaps between school-term and vacation dates.

Why consider CO as a traffic-related tracer? Near-road monitoring evidence finds higher CO near roadways, while particulate concentrations often follow the urban background and are less directly related to traffic volume (DeWinter et al., 2018, Atmospheric Environment 183, 94–105, doi:10.1016/j.atmosenv.2018.04.003). This motivates examining CO and NO2, while recognizing that no pollutant uniquely identifies school traffic.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** PRIMARY ANALYSIS: the conceptually motivated within-day double difference (07:00–09:00 arrival window vs the same-day 10:00–12:00 control window) with weekly-clustered inference, its weekday/weekend contrasts, the calendar-based falsification test, leave-one-zone-out and the residual diagnostics. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

output_dir = Path('../data_processed')
resultados_dir = output_dir / 'resultados_metodos'
graficos_dir = output_dir / 'graficos_metodos'
resultados_dir.mkdir(exist_ok=True)
graficos_dir.mkdir(exist_ok=True)

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'panel_pico_hibrido.csv',
                       output_dir / 'gaps_franjas_hibrido.csv',
                       output_dir / 'gaps_franjas_MIMA.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 3 files


## 2. SUR estimation with clustered errors

The five pollutant equations share regressors, so their SUR coefficients equal equation-by-equation OLS estimates. Their joint covariance is clustered by ISO week. The Wald test asks whether the school-term coefficient is zero for all five pollutants simultaneously.

### The mathematics of the double difference

The design's 2×2 table, expressed in mean log concentrations, is:

| | Arrival window 07–09 | Comparison window 10–12 |
|---|---|---|
| School-term dates | $\mu+\pi+\kappa+\beta$ | $\mu+\kappa$ |
| Vacation dates | $\mu+\pi$ | $\mu$ |

$\pi$ represents what distinguishes the arrival window regardless of calendar regime. $\kappa$ represents what distinguishes school-term dates in both windows, such as a shared seasonal component. $\beta$ is the additional arrival-window difference during school terms—the association of interest.

$$ \beta=\underbrace{(\text{arrival}-\text{comparison})_{\text{school term}}}_{\text{first difference}}-\underbrace{(\text{arrival}-\text{comparison})_{\text{vacation}}}_{\text{first difference}}. $$

Notebook 01 already computes the first subtraction in the gap variable. The regression performs the second:

$$ \text{gap}_{st}=\alpha+\beta\,\text{School}_t+\text{controls}+\varepsilon_{st}. $$

Thus, the school-term coefficient on the gap is the double difference.

### Clustered covariance and the joint test

The five equations have identical regressors. Estimating them as a SUR system therefore gives the same $\hat\beta_j$ as separate OLS regressions, but their COVARIANCE is estimated jointly. For weekly clusters $g=1,\ldots,G$:

$$ \hat V=A\left(\sum_{g=1}^G s_gs_g'\right)A'\frac{G}{G-1}\frac{n-1}{n-k},\qquad s_g=\operatorname{vec}(X_g'\hat U_g),\quad A=I_5\otimes(X'X)^{-1}. $$

$X_g$ and $\hat U_g$ contain the regressors and residuals for week $g$. The last two factors are the standard cluster finite-sample correction. Let $b$ contain the five school-term coefficients and let $V_b$ be their covariance block:

$$ W=b'V_b^{-1}b\sim\chi^2_5\quad\text{under }H_0:\ \beta_{CO}=\beta_{NO_2}=\beta_{O_3}=\beta_{PM_{10}}=\beta_{PM_{2.5}}=0. $$

One joint p-value addresses whether the calendar is associated with ANY part of the pollutant system. Pollutant-specific tests answer a different question and require attention to multiple testing.

### Hypotheses tested

- For each pollutant: $H_0:\beta_j=0$ against $H_1:\beta_j\ne0$.
- Joint Wald test: all five school-term coefficients equal zero, against at least one nonzero coefficient.
- Falsification: an artificial school indicator alternates by ISO-week parity within one calendar regime. A rejection raises concern about temporal structure. Consecutive weeks are not fully exchangeable, so failure to reject is limited support rather than proof of validity.

### Design assumptions and their diagnostics

No diagnostic establishes an assumption completely; each provides partial evidence.

1. **Valid pseudo-control.** Factors affecting both windows must cancel, or their morning evolution must not differ across regimes beyond the arrival-related association being studied. The hourly profile in notebook 09 and the falsification checks examine this, but cannot prove that 10:00–12:00 is a perfect control: weather, traffic and chemistry change during the morning.
2. **Measurement of the school-term indicator.** The indicator comes from the official SEP calendar, with DOF sources in notebook 00. It identifies calendar regimes, not attendance on every date. Teacher-council Fridays are not flagged; weekends within a term remain school-term dates and enter the pooled estimate. Weekday-only estimates are therefore reported separately. Attendance misclassification does not have a guaranteed direction of bias.
3. **No spillover between windows.** Arrival traffic should not affect the comparison window. For relatively inert primary pollutants such as CO, persistence into mid-morning would tend to attenuate the contrast; similar reasoning may apply to particulate matter. This direction is not guaranteed for ozone, whose morning concentration reflects NO titration and sunlight-driven photochemical formation.
4. **Limited dependence across weeks.** Inference clusters by ISO week. Section 5's Durbin–Watson statistics and residual ACF through 21 lags examine the strength and persistence of serial correlation. Notebook 08's wild bootstrap improves finite-sample inference for the chosen clusters, but does not protect against dependence between weeks.

In [2]:
def matriz_diseno(df, especificacion='preferida'):
    "\n    Build the design matrix: intercept, clase and fixed effects.\n    'preferida' = zone, month and weekday; 'sin_mes' = zone and weekday\n    (to assess month adjustment); 'estacion' = zone only.\n    "
    piezas = [pd.get_dummies(df['estacion'], prefix='est', drop_first=True).astype(float)]
    if especificacion == 'preferida':
        piezas.append(pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float))
    if especificacion in ('preferida', 'sin_mes'):
        piezas.append(pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float))
    X = pd.concat([pd.Series(1.0, index=df.index, name='const'),
                   df['clase'].astype(float)] + piezas, axis=1)
    return X

def sur_wald_cluster(df, X, columnas_y, cluster):
    '\n    Equation-wise OLS (identical regressors) and joint clustered covariance -> joint Wald test.\n    '
    Xv = X.values
    n, k = Xv.shape
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    Y = df[columnas_y].values
    B = XtX_inv @ Xv.T @ Y
    U = Y - Xv @ B
    J = len(columnas_y)
    grupos = df[cluster].values
    orden = np.argsort(grupos, kind='stable')
    Xg, Ug, gg = Xv[orden], U[orden], grupos[orden]
    S = np.zeros((J * k, J * k))
    inicios = np.flatnonzero(np.r_[True, gg[1:] != gg[:-1]])
    fines = np.r_[inicios[1:], len(gg)]
    G = len(inicios)
    for s, e in zip(inicios, fines):
        sg = (Xg[s:e].T @ Ug[s:e]).T.reshape(-1)
        S += np.outer(sg, sg)
    A = np.kron(np.eye(J), XtX_inv)
    V = A @ S @ A
    V *= (G / (G - 1)) * ((n - 1) / (n - k))
    idx_clase = X.columns.get_loc('clase')
    seleccion = [j * k + idx_clase for j in range(J)]
    b = B[idx_clase, :]
    Vb = V[np.ix_(seleccion, seleccion)]
    W = float(b @ np.linalg.inv(Vb) @ b)
    p_chi2 = float(1 - stats.chi2.cdf(W, J))
    resultados_por_var = {}
    for j, col in enumerate(columnas_y):
        se = float(np.sqrt(Vb[j, j]))
        z = b[j] / se
        resultados_por_var[col] = {
            'beta': round(float(b[j]), 4), 'se': round(se, 4),
            'pct': round((np.exp(b[j]) - 1) * 100, 2),
            'ci_lo_pct': round((np.exp(b[j] - 1.96 * se) - 1) * 100, 2),
            'ci_hi_pct': round((np.exp(b[j] + 1.96 * se) - 1) * 100, 2),
            'p': round(2 * (1 - stats.norm.cdf(abs(z))), 6)}
    return {'por_var': resultados_por_var, 'wald_chi2': round(W, 2),
            'p_chi2': round(p_chi2, 6), 'G_clusters': int(G), 'n': int(n), 'k': int(k)}

print('Estimator functions ready')

Estimator functions ready


## 3. Main estimation under both imputations

Alongside the main association, we calculate: (a) weekday-only and weekend-only estimates; (b) calendar-based falsification using alternating weeks within a regime; (c) leave-one-zone-out influence checks, reestimating six times and reporting the range; (d) the weekday-minus-weekend triple difference; and (e) the same estimator without month fixed effects.

Vacation dates occur in winter, Easter and summer, so comparing regimes without month adjustment can mix the school calendar with each pollutant's seasonal cycle. The falsification test examines only the alternating-week pattern; it does not exclude seasonal confounding, trends or differences among Christmas, Easter and summer breaks. Likewise, dropping one zone cannot detect a shock shared by all zones, such as heavy rain or city-wide changes in work or industrial activity.

The weekday–weekend contrast uses a fully interacted regression with separate nuisance coefficients, retaining subgroup covariance through shared ISO-week clusters.

In [3]:
columnas_gap = [f'gap_{c}' for c in contaminantes]
resultados = {}

for nombre_dataset in ['hibrido', 'MIMA']:
    gaps = pd.read_csv(f'../data_processed/gaps_franjas_{nombre_dataset}.csv', parse_dates=['date'])
    salida = {'n': len(gaps), 'gap_promedio_por_regimen': {}}
    for cl in (0, 1):
        sub = gaps[gaps['clase'] == cl]
        salida['gap_promedio_por_regimen'][f'clase_{cl}'] = {
            c: round(sub[f'gap_{c}'].mean(), 3) for c in contaminantes}

    X = matriz_diseno(gaps, 'preferida')
    salida['did_franjas'] = sur_wald_cluster(gaps, X, columnas_gap, 'wk')

    # Compare weekday/weekend estimates; the difference requires a formal test
    for nombre, sub in (('solo_entre_semana', gaps[gaps['finde'] == 0]),
                        ('solo_fin_de_semana', gaps[gaps['finde'] == 1])):
        Xs = matriz_diseno(sub, 'preferida')
        salida[nombre] = sur_wald_cluster(sub, Xs, columnas_gap, 'wk')

    # Falsification: artificial indicator from week parity within each regime
    for nombre, sub in (('placebo_vacaciones', gaps[gaps['clase'] == 0]),
                        ('placebo_clases', gaps[gaps['clase'] == 1])):
        s = sub.copy()
        s['clase'] = (s['wknum'] % 2).astype(int)
        Xs = matriz_diseno(s, 'preferida')
        salida[nombre] = sur_wald_cluster(s, Xs, columnas_gap, 'wk')

    # Leave-one-zone-out influence: refit after excluding each zone
    salida['leave_one_zone_out'] = {}
    for zona_fuera in sorted(gaps['estacion'].unique()):
        sub = gaps[gaps['estacion'] != zona_fuera]
        Xs = matriz_diseno(sub, 'preferida')
        r_l = sur_wald_cluster(sub, Xs, columnas_gap, 'wk')
        salida['leave_one_zone_out'][f'sin_{zona_fuera}'] = {
            'p_conjunto': r_l['p_chi2'],
            **{c: {k: r_l['por_var'][f'gap_{c}'][k] for k in ('pct', 'ci_lo_pct', 'ci_hi_pct', 'p')}
               for c in contaminantes}}

    # Contribution of month fixed effects: retain zone and weekday, omit month
    X_sin_mes = matriz_diseno(gaps, 'sin_mes')
    salida['sin_efectos_fijos_de_mes'] = sur_wald_cluster(gaps, X_sin_mes, columnas_gap, 'wk')

    # Fully interacted weekday/weekend regression: independent nuisance slopes,
    # but shared weekly clusters retain covariance between the subgroup estimates.
    wd = gaps['finde'].eq(0).to_numpy()
    Xwd = matriz_diseno(gaps.loc[wd], 'preferida').to_numpy(dtype=float)
    Xwe = matriz_diseno(gaps.loc[~wd], 'preferida').to_numpy(dtype=float)
    k_wd = Xwd.shape[1]
    Xjoint = np.zeros((len(gaps), k_wd + Xwe.shape[1]))
    Xjoint[wd, :k_wd] = Xwd
    Xjoint[~wd, k_wd:] = Xwe
    Yjoint = gaps[columnas_gap].to_numpy(dtype=float)
    bread = np.linalg.pinv(Xjoint.T @ Xjoint)
    Bjoint = bread @ Xjoint.T @ Yjoint
    resid = Yjoint - Xjoint @ Bjoint
    contrast = np.zeros(Xjoint.shape[1])
    # The design begins with const, clase, followed by the fixed effects.
    contrast[1], contrast[k_wd + 1] = 1.0, -1.0
    delta = contrast @ Bjoint
    influence = (Xjoint @ bread @ contrast)[:, None] * resid
    cluster_ids, unique_weeks = pd.factorize(gaps['wk'])
    G = len(unique_weeks)
    scores = np.zeros((G, Yjoint.shape[1]))
    np.add.at(scores, cluster_ids, influence)
    n, k = Xjoint.shape
    Vcontrast = scores.T @ scores * G / (G - 1) * (n - 1) / (n - k)
    se = np.sqrt(np.diag(Vcontrast))
    salida['triple_dif_wd_menos_we'] = {
        c: {'delta_beta': round(float(delta[j]), 4),
            'se_cluster': round(float(se[j]), 4),
            'p_cluster': round(float(2 * stats.norm.sf(abs(delta[j] / se[j]))), 4),
            'G_clusters': G,
            'metodo': 'fully interacted regression; shared ISO-week clusters'}
        for j, c in enumerate(contaminantes)}

    resultados[nombre_dataset] = salida

    print("=" * 70)
    print(f"WITHIN-DAY GAP DIFFERENCE-IN-DIFFERENCES — dataset {nombre_dataset}")
    print("=" * 70)
    did = salida['did_franjas']
    print(f"Joint effect: chi2 = {did['wald_chi2']} | p = {did['p_chi2']} "
          f"(G = {did['G_clusters']} weeks, n = {did['n']})")
    for c in contaminantes:
        e = did['por_var'][f'gap_{c}']
        print(f"  {c}: {e['pct']:+.2f}% [{e['ci_lo_pct']}, {e['ci_hi_pct']}] p = {e['p']}")
    print(f"Weekday p = {salida['solo_entre_semana']['p_chi2']} | "
          f"weekend p = {salida['solo_fin_de_semana']['p_chi2']}")
    print(f"Artificial indicator within vacation p = {salida['placebo_vacaciones']['p_chi2']} | "
          f"within school term p = {salida['placebo_clases']['p_chi2']}")
    lozo_co = [r_l['CO']['pct'] for r_l in salida['leave_one_zone_out'].values()]
    lozo_p = [r_l['p_conjunto'] for r_l in salida['leave_one_zone_out'].values()]
    print(f"Leave-one-zone-out (CO): effect between {min(lozo_co):+.1f}% and {max(lozo_co):+.1f}% | "
          f"joint p between {min(lozo_p):.4f} y {max(lozo_p):.4f}")
    print('Without month fixed effects (zone + weekday), compared with the preferred specification:')
    for c in contaminantes:
        e0 = salida['sin_efectos_fijos_de_mes']['por_var'][f'gap_{c}']
        e1 = did['por_var'][f'gap_{c}']
        print(f"  {c}: without month {e0['pct']:+.1f}% (p = {e0['p']}) -> with month {e1['pct']:+.1f}% (p = {e1['p']})")

with open(resultados_dir / 'did_franjas.json', 'w') as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print(f"\nSaved to: {resultados_dir / 'did_franjas.json'}")

WITHIN-DAY GAP DIFFERENCE-IN-DIFFERENCES — dataset hibrido
Joint effect: chi2 = 24.11 | p = 0.000207 (G = 158 weeks, n = 6576)
  CO: +4.56% [1.49, 7.71] p = 0.003312
  NO2: +1.02% [-6.15, 8.74] p = 0.786724
  O3: -2.61% [-12.67, 8.61] p = 0.634717
  PM10: -0.54% [-5.68, 4.89] p = 0.842731
  PM2.5: +0.52% [-4.45, 5.75] p = 0.84075
Weekday p = 3.4e-05 | weekend p = 0.052398
Artificial indicator within vacation p = 0.046307 | within school term p = 0.506341
Leave-one-zone-out (CO): effect between +3.9% and +5.1% | joint p between 0.0001 y 0.0065
Without month fixed effects (zone + weekday), compared with the preferred specification:
  CO: without month +6.3% (p = 0.0) -> with month +4.6% (p = 0.003312)
  NO2: without month +6.5% (p = 0.018969) -> with month +1.0% (p = 0.786724)
  O3: without month +5.6% (p = 0.290781) -> with month -2.6% (p = 0.634717)
  PM10: without month +1.8% (p = 0.315052) -> with month -0.5% (p = 0.842731)
  PM2.5: without month +9.0% (p = 0.008481) -> with month +0

WITHIN-DAY GAP DIFFERENCE-IN-DIFFERENCES — dataset MIMA
Joint effect: chi2 = 21.83 | p = 0.000565 (G = 158 weeks, n = 6576)
  CO: +4.37% [1.3, 7.54] p = 0.005038
  NO2: +0.76% [-6.55, 8.64] p = 0.844536
  O3: -2.11% [-12.13, 9.07] p = 0.699551
  PM10: -0.70% [-5.83, 4.7] p = 0.793762
  PM2.5: +1.29% [-4.0, 6.88] p = 0.639052
Weekday p = 2.7e-05 | weekend p = 0.080544
Artificial indicator within vacation p = 0.053109 | within school term p = 0.4621
Leave-one-zone-out (CO): effect between +3.7% and +5.0% | joint p between 0.0001 y 0.0135
Without month fixed effects (zone + weekday), compared with the preferred specification:
  CO: without month +6.1% (p = 0.0) -> with month +4.4% (p = 0.005038)
  NO2: without month +6.2% (p = 0.027876) -> with month +0.8% (p = 0.844536)
  O3: without month +6.0% (p = 0.262577) -> with month -2.1% (p = 0.699551)
  PM10: without month +1.8% (p = 0.308701) -> with month -0.7% (p = 0.793762)
  PM2.5: without month +9.2% (p = 0.006646) -> with month +1.3% (p 

## 4. Estimated association by pollutant

In [4]:
fig, ax = plt.subplots(figsize=(13, 6.5))

did_h = resultados['hibrido']['did_franjas']['por_var']
did_m = resultados['MIMA']['did_franjas']['por_var']

x = np.arange(len(contaminantes))
width = 0.35
for desplazamiento, did, etiqueta, color in [(-width/2, did_h, 'Hybrid imputation', '#3498db'),
                                             (width/2, did_m, 'MIMA imputation', '#e74c3c')]:
    valores = [did[f'gap_{c}']['pct'] for c in contaminantes]
    err_lo = [did[f'gap_{c}']['pct'] - did[f'gap_{c}']['ci_lo_pct'] for c in contaminantes]
    err_hi = [did[f'gap_{c}']['ci_hi_pct'] - did[f'gap_{c}']['pct'] for c in contaminantes]
    bars = ax.bar(x + desplazamiento, valores, width, yerr=[err_lo, err_hi],
                  label=etiqueta, color=color, alpha=0.8, edgecolor='black', capsize=4)
    for bar, c in zip(bars, contaminantes):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + (0.6 if bar.get_height() >= 0 else -2.2),
                f"{did[f'gap_{c}']['pct']:+.1f}%",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(contaminantes)
ax.set_xlabel('Pollutant', fontsize=12, fontweight='bold')
ax.set_ylabel('School-term change in the morning/control ratio (%)', fontsize=12, fontweight='bold')
ax.set_title('Within-day difference-in-differences: school-term change in the 7-9 vs 10-12 ratio\n(percent effect with transformed 95% CI; weekly-clustered errors)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(graficos_dir / '01_did_franjas_efectos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {graficos_dir / '01_did_franjas_efectos.png'}")

Plot saved to: ../data_processed/graficos_metodos/01_did_franjas_efectos.png


## 5. Residual diagnostics: Durbin–Watson and autocorrelation

We examine how much autocorrelation remains. Diagnostics provide partial evidence, not proof that dependence has disappeared. For residuals ordered by date:

$$ DW=\frac{\sum_{t=2}^n(\hat\varepsilon_t-\hat\varepsilon_{t-1})^2}{\sum_{t=1}^n\hat\varepsilon_t^2}\approx2(1-\hat\rho_1). $$

DW is near 2 without first-order correlation, below 2 with positive correlation and above 2 with negative correlation. It is calculated separately for each monitoring unit: concatenating units would create artificial jumps. Comparing LEVEL and GAP models shows whether within-day subtraction reduces the persistence seen in daily series.

Implementation note: fixed effects enter as indicator variables. Equivalent R specifications are `lm(y ~ clase + factor(estacion) + factor(mes) + factor(dow))` or `fixest::feols(y ~ clase | estacion + mes + dow, cluster = ~semana)`. Python implementations can use statsmodels with the same dummies, as here, or appropriately specified panel estimators. Matching regressors, samples and covariance corrections is essential. The Spanish identifiers in these examples are the actual data-column names.

In [5]:
from statsmodels.stats.stattools import durbin_watson

panel_diag = pd.read_csv('../data_processed/panel_pico_hibrido.csv', parse_dates=['date'])
gaps_diag = pd.read_csv('../data_processed/gaps_franjas_hibrido.csv', parse_dates=['date'])

def residuos_por_estacion(df, columna_y):
    '\n    OLS y ~ clase + month + weekday within ONE unit; residuals in chronological order.\n    '
    df = df.dropna(subset=[columna_y]).sort_values('date')
    X = pd.concat([pd.Series(1.0, index=df.index, name='const'), df['clase'].astype(float),
                   pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float),
                   pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float)], axis=1)
    beta, *_ = np.linalg.lstsq(X.values, df[columna_y].values, rcond=None)
    return df[columna_y].values - X.values @ beta

filas_dw = []
for estacion_m in sorted(panel_diag['estacion'].unique()):
    fila = {'estacion': estacion_m}
    for c in contaminantes:
        r_nivel = residuos_por_estacion(panel_diag[panel_diag['estacion'] == estacion_m], f'l_{c}')
        r_gap = residuos_por_estacion(gaps_diag[gaps_diag['estacion'] == estacion_m], f'gap_{c}')
        fila[f'{c}_nivel'] = round(durbin_watson(r_nivel), 2)
        fila[f'{c}_gap'] = round(durbin_watson(r_gap), 2)
    filas_dw.append(fila)
tabla_dw = pd.DataFrame(filas_dw)

print("=" * 70)
print('DURBIN-WATSON BY UNIT (log level vs gap; ~2 suggests little first-order correlation)')
print("=" * 70)
print(tabla_dw.to_string(index=False))
tabla_dw.to_csv(resultados_dir / 'durbin_watson_por_estacion.csv', index=False)

dw_nivel_medio = tabla_dw[[f'{c}_nivel' for c in contaminantes]].values.mean()
dw_gap_medio = tabla_dw[[f'{c}_gap' for c in contaminantes]].values.mean()
print(f"\nMean DW for levels: {dw_nivel_medio:.2f} | mean DW for gaps: {dw_gap_medio:.2f}")
print('=> Subtracting windows reduces first-order residual autocorrelation')
print('   (gap statistics are closer to 2), but DW does not establish its elimination')
print('   or assess longer lags; the ACF below examines those lags.')
print(f"\nSaved to: {resultados_dir / 'durbin_watson_por_estacion.csv'}")

DURBIN-WATSON BY UNIT (log level vs gap; ~2 suggests little first-order correlation)
 estacion  CO_nivel  CO_gap  NO2_nivel  NO2_gap  O3_nivel  O3_gap  PM10_nivel  PM10_gap  PM2.5_nivel  PM2.5_gap
   CENTRO      0.62    1.88       1.35     1.72      1.10    1.30        1.02      1.86         1.22       1.86
 NORESTE3      0.34    1.41       1.18     1.58      1.48    1.39        1.10      1.92         1.28       1.60
NOROESTE3      0.39    1.61       1.01     1.61      1.46    1.33        1.06      1.86         1.02       1.63
   NORTE2      1.01    1.70       1.03     1.60      1.20    1.26        1.11      1.86         1.31       1.72
      SUR      0.39    1.56       0.86     1.40      1.49    1.41        0.98      1.72         1.17       1.56
SUROESTE2      0.49    1.39       1.19     1.64      1.11    1.37        1.14      1.93         1.39       1.85

Mean DW for levels: 1.05 | mean DW for gaps: 1.62
=> Subtracting windows reduces first-order residual autocorrelation
   (gap stat

### Residual autocorrelation function (ACF)

DW examines adjacent residuals only. To assess how long dependence persists, compute $\hat\rho_k=\operatorname{corr}(\hat\varepsilon_t,\hat\varepsilon_{t-k})$ for $k=1,\ldots,21$ days, separately by zone, for levels and gaps. We examine lag 1, whether correlation is small by lag 7 and what remains beyond a week. Clear dependence beyond one week would challenge weekly clustering; the wild bootstrap with the same clusters would not solve that problem.

In [6]:
from statsmodels.tsa.stattools import acf

rezagos = 21
filas_acf = []
for estacion_m in sorted(panel_diag['estacion'].unique()):
    for c in contaminantes:
        for tipo, df_d, col in (('nivel', panel_diag, f'l_{c}'), ('gap', gaps_diag, f'gap_{c}')):
            r = residuos_por_estacion(df_d[df_d['estacion'] == estacion_m], col)
            rho = acf(r, nlags=rezagos, fft=True)
            for k in range(1, rezagos + 1):
                filas_acf.append({'estacion': estacion_m, 'contaminante': c, 'serie': tipo,
                                  'rezago': k, 'acf': round(float(rho[k]), 4)})
tabla_acf = pd.DataFrame(filas_acf)
tabla_acf.to_csv(resultados_dir / 'acf_residuos_por_estacion.csv', index=False)

# Average ACF over zones/pollutants by lag; reference band 2/sqrt(n)
n_obs = int(panel_diag.groupby('estacion').size().mean())
banda = 2 / np.sqrt(n_obs)
resumen_acf = (tabla_acf.groupby(['serie', 'rezago'])['acf'].mean().unstack('serie')
               [['nivel', 'gap']].round(3))
resumen_acf.to_csv(resultados_dir / 'acf_residuos_promedio.csv')

print("=" * 70)
print('MEAN RESIDUAL ACF (zones and pollutants pooled; reference band ~ ±%.3f)' % banda)
print("=" * 70)
print(resumen_acf.loc[[1, 2, 3, 5, 7, 10, 14, 21]].to_string())
max_gap_semana = float(tabla_acf[(tabla_acf['serie'] == 'gap') & (tabla_acf['rezago'] >= 8)]['acf'].abs().max())
print(f"\nGap: lag 1 = {resumen_acf.loc[1, 'gap']:.3f}, lag 7 = {resumen_acf.loc[7, 'gap']:.3f}, "
      f"rezago 14 = {resumen_acf.loc[14, 'gap']:.3f}; maximum individual |ACF| beyond one week = {max_gap_semana:.3f}")
print(f"Level: lag 1 = {resumen_acf.loc[1, 'nivel']:.3f}, lag 7 = {resumen_acf.loc[7, 'nivel']:.3f}, "
      f"rezago 14 = {resumen_acf.loc[14, 'nivel']:.3f}")
print('=> Within-day subtraction reduces dependence but does not eliminate it.')
print('   Weekly clustering allows within-week dependence; the wild cluster bootstrap')
print('   (notebook 08) does not by itself address dependence across weeks.')

MEAN RESIDUAL ACF (zones and pollutants pooled; reference band ~ ±0.060)
serie   nivel    gap
rezago              
1       0.474  0.190
2       0.298  0.090
3       0.239  0.059
5       0.180  0.034
7       0.173  0.024
10      0.154  0.019
14      0.115  0.002
21      0.113  0.003

Gap: lag 1 = 0.190, lag 7 = 0.024, rezago 14 = 0.002; maximum individual |ACF| beyond one week = 0.150
Level: lag 1 = 0.474, lag 7 = 0.173, rezago 14 = 0.115
=> Within-day subtraction reduces dependence but does not eliminate it.
   Weekly clustering allows within-week dependence; the wild cluster bootstrap
   (notebook 08) does not by itself address dependence across weeks.


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, tipo, titulo in zip(axes, ['nivel', 'gap'], ['LEVEL residuals (log 7-9)', 'GAP residuals (7-9 minus 10-12)']):
    for c in contaminantes:
        sub = (tabla_acf[(tabla_acf['serie'] == tipo) & (tabla_acf['contaminante'] == c)]
               .groupby('rezago')['acf'].mean())
        ax.plot(sub.index, sub.values, marker='o', markersize=3.5, linewidth=1.8, label=c)
    ax.axhline(0, color='black', linewidth=1)
    ax.axhline(banda, color='gray', linestyle='--', linewidth=1)
    ax.axhline(-banda, color='gray', linestyle='--', linewidth=1)
    ax.axvline(7, color='#e74c3c', linestyle=':', linewidth=1.2)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Lag (days)', fontsize=11, fontweight='bold')
    ax.set_xticks(range(0, rezagos + 1, 7))
axes[0].set_ylabel('Residual autocorrelation', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
fig.suptitle('Residual ACF by lag (average over six zones; red line = one week; gray band ~ 2/sqrt(n))',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(graficos_dir / '05_acf_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to: {resultados_dir / 'acf_residuos_por_estacion.csv'} y {resultados_dir / 'acf_residuos_promedio.csv'}")
print(f"Plot saved to: {graficos_dir / '05_acf_residuos.png'}")

Saved to: ../data_processed/resultados_metodos/acf_residuos_por_estacion.csv y ../data_processed/resultados_metodos/acf_residuos_promedio.csv
Plot saved to: ../data_processed/graficos_metodos/05_acf_residuos.png


## 6. Summary and interpretation

In [8]:
print("=" * 70)
print('INTERPRETING THE WITHIN-DAY DESIGN (official SEP calendar)')
print("=" * 70)
salida_h = resultados['hibrido']
triple = salida_h['triple_dif_wd_menos_we']
co = salida_h['did_franjas']['por_var']['gap_CO']
co_wd = salida_h['solo_entre_semana']['por_var']['gap_CO']
co_we = salida_h['solo_fin_de_semana']['por_var']['gap_CO']
p_conj = salida_h['did_franjas']['p_chi2']
individuales = [c for c in contaminantes if salida_h['did_franjas']['por_var'][f'gap_{c}']['p'] < 0.05]
print(f"1. The joint effect {'rejects' if p_conj < 0.05 else 'does not reject'} zero (p = {p_conj}).")
print(f"2. Individually significant associations: {', '.join(individuales) if individuales else 'none'}."
      f" CO: {co['pct']:+.1f}% [{co['ci_lo_pct']}, {co['ci_hi_pct']}] (p = {co['p']}).")
print(f"3. Weekday CO {co_wd['pct']:+.1f}% (p = {co_wd['p']}) versus {co_we['pct']:+.1f}% on")
print(f"   weekends (p = {co_we['p']}); weekday-weekend coefficient difference = {triple['CO']['delta_beta']:+.3f},")
print(f"   p = {triple['CO']['p_cluster']} ({'significant' if triple['CO']['p_cluster'] < 0.05 else 'borderline' if triple['CO']['p_cluster'] < 0.10 else 'not significant'}).")
p_we = salida_h['solo_fin_de_semana']['p_chi2']
we_sig = [c for c in contaminantes if salida_h['solo_fin_de_semana']['por_var'][f'gap_{c}']['p'] < 0.05]
if p_we < 0.05:
    print(f"4. WARNING: the weekend joint test also rejects (p = {p_we}), with")
    print(f"   individually significant pollutants: {', '.join(we_sig) if we_sig else 'none'}.")
    print(f"   Weekend changes cannot be attributed to school arrival; other activity may")
    print(f"   differ between term and vacation periods (general mobility and travel).")
else:
    print(f"4. The weekend joint test does not reject (p = {p_we}); this alone does not establish")
    print(f"   a weekday-weekend difference or isolate school traffic.")
p_pc, p_pv = salida_h['placebo_clases']['p_chi2'], salida_h['placebo_vacaciones']['p_chi2']
def _lee(p):
    return 'limpia' if p >= 0.10 else 'borderline' if p >= 0.05 else 'REJECTS (inspect)'
print(f"5. Falsification tests (artificial term indicator): within school term {_lee(p_pc)} (p = {p_pc});")
print(f"   within vacation {_lee(p_pv)} (p = {p_pv}; with MIMA {resultados['MIMA']['placebo_vacaciones']['p_chi2']}).")
print(f"   Rejection by the calendar-based falsification test within vacations warns of residual temporal structure; it does not validate the design.")
print(f"6. Both imputation methods show a similar CO association (MIMA: "
      f"{resultados['MIMA']['did_franjas']['por_var']['gap_CO']['pct']:+.1f}%).")
sin_mes = salida_h['sin_efectos_fijos_de_mes']['por_var']
movidos = [c for c in contaminantes
           if abs(sin_mes[f'gap_{c}']['pct'] - salida_h['did_franjas']['por_var'][f'gap_{c}']['pct']) >= 3]
print(f"7. Month fixed effects change the estimate by at least 3 percentage points for: {', '.join(movidos) if movidos else 'none'}.")
for c in movidos:
    print(f"   {c}: {sin_mes[f'gap_{c}']['pct']:+.1f}% without month -> "
          f"{salida_h['did_franjas']['por_var'][f'gap_{c}']['pct']:+.1f}% with month")
if movidos:
    print(f"   Month adjustment accounts for some seasonal composition in the raw diurnal curves.")
    print(f"   CO retains a positive morning/control contrast in the adjusted main specification.")

print('\nCOMPLETED')

INTERPRETING THE WITHIN-DAY DESIGN (official SEP calendar)
1. The joint effect rejects zero (p = 0.000207).
2. Individually significant associations: CO. CO: +4.6% [1.49, 7.71] (p = 0.003312).
3. Weekday CO +5.5% (p = 0.003029) versus +1.4% on
   weekends (p = 0.467087); weekday-weekend coefficient difference = +0.040,
   p = 0.1075 (not significant).
4. The weekend joint test does not reject (p = 0.052398); this alone does not establish
   a weekday-weekend difference or isolate school traffic.
5. Falsification tests (artificial term indicator): within school term limpia (p = 0.506341);
   within vacation REJECTS (inspect) (p = 0.046307; with MIMA 0.053109).
   Rejection by the calendar-based falsification test within vacations warns of residual temporal structure; it does not validate the design.
6. Both imputation methods show a similar CO association (MIMA: +4.4%).
7. Month fixed effects change the estimate by at least 3 percentage points for: NO2, O3, PM2.5.
   NO2: +6.5% without 